# Statistical power and sample size 
## Introduction


## Definitions
### Statistical power


### The effect size


## Relationship between power, effect size, sample size, and variability
### One-sample t-test


In [ ]:

import matplotlib.pyplot as plt
from statsmodels.stats.power import TTestPower

# Simulation parameters
diff = -10           # Mean level DECREASE compared to known population mean
std_devs = [20, 40]  # Different levels of standard deviation
power = 0.8          # Desired power (80%)
α = 0.05             # Significance level
max_sample_size = 151  # Highest sample size to test

# Calculate power curves
power_analysis = TTestPower()

power_curves = {}
for sd in std_devs:
    power_curves[sd] = [
        power_analysis.power(
            effect_size=diff/sd,
            alpha=α,
            nobs=n,
            alternative='smaller',
        ) for n in range(2, max_sample_size)]  # Start from 2 observations

# Calculate required sample sizes for each SD
required_sample_sizes = {}
for sd in std_devs:
    required_sample_sizes[sd] = power_analysis.solve_power(
        effect_size=diff/sd,
        power=power,
        alpha=α,
        nobs=None, # this is the one we want to solve
        alternative='smaller')

# Plotting
plt.figure(figsize=(3.5, 3))
for sd, powers in power_curves.items():
    plt.plot(
        range(2, max_sample_size),
        powers,
        label=f'σ={sd}', lw=2)

plt.axhline(
    y=power,
    ls=':', c='red',
    label=f'Desired power ({power*100:n}%)')

for sd, powers in required_sample_sizes.items():
    plt.axvline(
        required_sample_sizes[sd],
        linestyle='--',color='gray')

plt.xlabel("Sample size (n)")
plt.ylabel("Statistical power (1 - β)")
plt.title("Power curves (one-sample t-test)")
plt.xlim(0, max_sample_size+5)
plt.ylim(0, 1.05)
plt.legend();

### Unpaired t-test


In [ ]:

import numpy as np
from statsmodels.stats.power import TTestIndPower

# Analysis setup
power_analysis = TTestIndPower()
effect_sizes = np.array([0.2, 0.5, 0.8])  # List of effect sizes
sample_sizes = np.arange(5, 401, 10)      # List of sample sizes
α = 0.05
power = 0.8  # Or β = 0.2

# Calculate required sample sizes for each SD
required_sample_sizes = {}
for es in effect_sizes:
    required_sample_sizes[es] = power_analysis.solve_power(
        effect_size=es,
        power=power,
        alpha=α,
        # Arguments are a little different than in TTestPower
        ratio=1,     # Ratio nobs2:nobs1
        nobs1=None,  # Argument we want to solve
        alternative='two-sided')

fig, ax = plt.subplots(1, 1, figsize=(3.5, 3))
# Plotting with plot_power function
power_analysis.plot_power(
    dep_var='nobs',
    nobs=sample_sizes,
    effect_size=effect_sizes,
    alpha=α,
    ax=ax)

plt.axhline(
    y=power,
    ls=':', c='red',
    label=f'Desired power ({power*100:.0f}%)')

for es, powers in required_sample_sizes.items():
    plt.axvline(
        required_sample_sizes[es],
        linestyle='--',color='gray')

plt.xlabel('Sample size (n) per group')
plt.ylabel('Power (1 - β)')
plt.xlim(0, max(sample_sizes)+5)
plt.title('Power analysis (independent t-test)')
plt.legend();

In [ ]:
#| fig-subcap:
#|   - "Power vs. sample size (effect size = 0.5)"
#|   - "Detectable mean difference (power = 80%)"
#| layout-ncol: 2

# Parameters
mean_difference = 5  # Mean difference tested
sd = 10              # Standard deviation
alpha = 0.05
power = 0.8
sample_sizes = np.arange(10, 101, 10) # Range of sample sizes per group

# Power analysis for subplot 1
power_analysis = TTestIndPower()
powers = [
    power_analysis.power(
        effect_size=mean_difference/sd,
        alpha=alpha,
        nobs1=n,
        ratio=1,
    ) for n in sample_sizes]

# Calculate mean differences for subplot 2
mean_diffs = [
    power_analysis.solve_power(
        effect_size=None,
        power=power,
        alpha=alpha,
        nobs1=n,
        ratio=1.0,
    # multiply the solved effect size by SD to get the mean difference
    ) * sd for n in sample_sizes]

# Power vs. Sample Size
plt.figure(figsize=(3.5, 3))
plt.plot(
    sample_sizes, powers,
    'o-', color='blue', linewidth=2)
plt.xlabel('Sample size (n) per group')
plt.ylabel('Power (1 - β)')
plt.title(
    f"Power vs. sample size")
plt.xlim(0, max(sample_sizes)+5)
plt.ylim(0, 1.05)
plt.show()

# Mean Difference vs. Sample Size
plt.figure(figsize=(3.5, 3))
plt.plot(
    sample_sizes, mean_diffs,
    'o-', color='green', linewidth=2)
plt.xlabel('Sample size (n) per group')
plt.ylabel('Detectable mean difference')
plt.title(
    f"Detectable effect vs. sample size")
plt.xlim(0, max(sample_sizes)+5)
plt.ylim(0, 15)
plt.show()

### Paired t-test


In [ ]:

import pingouin as pg

# Parameters
effect_sizes = np.array([0.2, 0.5, 0.8])  # Effect sizes (Cohen's d)
α = 0.05
sample_sizes = np.arange(5, 101, 5)       # Number of pairs to consider
power = 0.8

# Calculate and print required sample sizes
print("Required sample sizes for 80% power in a paired t-test:")
for es in effect_sizes:
    required_n = pg.power_ttest(
        d=es,
        n=None, # parameter to solve
        power=power,
        alpha=α,
        contrast='paired',
        alternative='two-sided')
    print(f"Effect size = {es}: {int(np.ceil(required_n))} pairs")

# Plotting
plt.figure(figsize=(3.5, 3))
for es in effect_sizes:
    power_values = pg.power_ttest(
        d=es,
        n=sample_sizes,
        power=None, # parameter we want to solve
        alpha=α,
        contrast='paired',
        alternative='two-sided', # default
    )

    plt.plot(
        sample_sizes,
        power_values,
        lw=2,
        # 'o-',
        label=f'Effect size = {es:.1f}')

plt.axhline(
    y=power,
    ls=':', c='red',
    label=f'Desired power ({power*100:.0f}%)')

plt.xlabel('Sample size (paired observations)')
plt.ylabel('Power (1 - β)')
plt.title("Power analysis (paired t-test)")
plt.grid(axis='y', linestyle='--')
plt.xlim(0, max(sample_sizes)+2)
plt.ylim(0, 1.05)
plt.legend();

## Choosing a sample size
### The choice of 80% power
### Tools for power calculations


In [ ]:
from statsmodels.stats.power import tt_ind_solve_power

# Parameters
mean_difference = 17            # The difference between the means
pooled_standard_deviation = 17  # The pooled standard deviation
effect_size = mean_difference / pooled_standard_deviation  # Cohen's d
α = 0.05
desired_power = 0.8
ratio = 1.0                     # Sample size ratio (equal sample sizes)

# Calculate sample size (directly)
n_per_group = tt_ind_solve_power(
    effect_size=effect_size,
    alpha=α,
    power=desired_power,
    ratio=ratio,
    alternative='two-sided')

# Display result (verbose)
print("Power analysis for independent two-sample t-test")
print("------------------------------------------------")
print(f"Mean difference: {mean_difference}")
print(f"Pooled standard deviation: {pooled_standard_deviation}")
print(f"Effect size (Cohen's d): {effect_size:.2f}")
print(f"Significance level (α): {α}")
print(f"Desired power (1 - β): {desired_power}")
print(f"Ratio of sample sizes (n2/n1): {ratio}")
print(
    f"\nRequired sample size per group is: \
{int(np.ceil(n_per_group))} participants")

### The simulation-based approach


In [ ]:
from scipy.stats import ttest_ind

# For reproducibility
np.random.seed(111)

def simulate_study(
        n, mean_difference=0, control_mean=130,
        control_sd=15, treatment_sd=15, alpha=.05):
    """
    Simulates a study with two independent groups (control and treatment)
    and performs an independent two-sample t-test.

    Args:
        n: sample size per group.
        mean_difference: difference between the means of the groups. 
            Defaults to 0.
        control_mean: mean of the control group. Defaults to 130.
        control_sd: standard deviation of the control group.
            Defaults to 15.
        treatment_sd: standard deviation of the treatment group. 
            Defaults to 15.
        alpha: significance level. Defaults to 0.05.

    Returns:
        bool: True if the P value is less than alpha (significant), 
            False otherwise.
    """
    # Generate the normal distributions
    control_group = np.random.normal(control_mean, control_sd, n)
    treatment_group = np.random.normal(
        control_mean-mean_difference, treatment_sd, n)

    # Perform a t-test and grab the P value
    _, p_value = ttest_ind(control_group, treatment_group)

    # Return if P value is significant or not
    return p_value < alpha

# Simulation parameters
mean_difference = 10    # Difference between the means
α = 0.05                # Significance level
desired_power = 0.8     # Target power
num_simulations = 1000  # Number of simulated studies
max_sample_size = 80    # Maximum sample size per group
control_mean = 130      # Mean of control group
control_sd = 15         # Standard deviation of control group
treatment_sd = 15       # Standard deviation of treatment group

# Run the simulations
results = []  # Store the results in a list
print("Simulation of independent two-sample t-tests for power analysis")
print("---------------------------------------------------------------")
print(f"Mean difference: {mean_difference}")
print(f"Significance level (α): {α}")
print(f"Desired power: {desired_power}")
print(f"Number of simulations: {num_simulations}")
print(f"Maximum sample size per group: {max_sample_size}")
print(f"Control group Mean: {control_mean}")
print(f"Control group standard deviation: {control_sd}")
print(f"Treatment group standard deviation: {treatment_sd}")
print("---------------------------------------------------------------")
print("Running simulations...")  # This makes professional doesn't it?

# Iterate over sample sizes, starting from 10, 
# up to the max_sample_size (steps of 5)
for n in range(10, max_sample_size, 5):
    # Simulate num_simulations studies for the current sample size (n)
    signif_results = sum(simulate_study(
        n,
        mean_difference=mean_difference,
        control_mean=control_mean,
        control_sd=control_sd,
        treatment_sd=treatment_sd,
        alpha=α) for _ in range(num_simulations))

    # Calculate the power as the proportion of significant results
    power = signif_results / num_simulations

    # Store the current sample size and the calculated power
    results.append((n, power))
    print(f"Sample Size per Group: {n}, Power: {power:.4f}")

print("---------------------------------------------------------------")

# Find the smallest sample size that achieves 80% power
for n, power in results:
    if power >= desired_power:
        print(f"Required sample size per group to achieve a power >= \
{desired_power:.2f} is: {n}")
        break
else:
    print(f"The desired power of {desired_power:.2f} was not achieved \
within the tested sample sizes.")

### Comparing proportions (A/B test)


In [ ]:
from scipy.stats import norm
from statsmodels.stats.proportion import (
    samplesize_proportions_2indep_onetail)

# Parameters
baseline_proportion = 5 / 100  # Baseline (e.g., complication rate)
desired_proportion = 3 / 100   # Desired (i.e., 2% absolute change)

# Absolute difference between proportions
absolute_difference = desired_proportion - baseline_proportion
α = 0.05
desired_power = 0.9
ratio = 1.0

# Calculate sample size using manual calculations
za = norm.ppf(1 - (α / 2))  # Z-score for alpha/2 (two-sided test)
zb = np.abs(norm.ppf(1 - desired_power))  # Z-score for power

numerator_manual = (za + zb)** 2 * (
    desired_proportion * (1 - desired_proportion)
    + baseline_proportion * (1 - baseline_proportion))
denominator_manual = (desired_proportion - baseline_proportion)**2
n_per_group_manual = numerator_manual / denominator_manual

# Calculate sample size using statsmodels
n_per_group_statsmodels = samplesize_proportions_2indep_onetail(
    diff=absolute_difference,   # Difference between proportions
    prop2=baseline_proportion,  # Baseline proportion
    power=desired_power,
    ratio=ratio,
    alpha=α)

# Display results (verbose)
print("Sample size for comparing two proportions")
print("-----------------------------------------")
print(f"Baseline proportion (p2): {baseline_proportion:.4f}")
print(f"Desired proportion (p1): {desired_proportion:.4f}")
print(f"Absolute difference (p1 - p2): {absolute_difference:.4f}")
print(f"Significance level (α): {α}")
print(f"Desired power (1 - β): {desired_power}")
print(f"Ratio of sample sizes (n2/n1): {ratio}")
print("-----------------------------------------")
print("Manual calculation (formula):")
print(f" Z-score (α/2): {za:.4f}")
print(f" Z-score (power): {zb:.4f}")
print(f" Numerator: {numerator_manual:.4f}")
print(f" Denominator: {denominator_manual:.4f}")
print(f" Required sample size per group: \
{int(np.ceil(n_per_group_manual))}")
print()
print("statsmodels:")
print(f" Required sample size per group: \
{int(np.ceil(n_per_group_statsmodels))}")

## Complexity when computing sample size
### Incorporating FPRP into sample size decisions


In [ ]:
alphas = [.05, .01, .001]     # Significance levels tested
powers = [.80, .90, .95, .99] # Power levels tested
probas= [0.01, 0.1, .25, .5]  # Estimates of prior probability

print("FPRP affected by α, power, and prior probability\n")

print(
    f"α\tpower\t{100*probas[0]}%\t{100*probas[1]}%\
\t{100*probas[2]}%\t{100*probas[3]}%")
print("-"*48, end='')

for alpha in alphas:
    print(f"\n{alpha:4.3f}", end="")

    for power in powers:
        print(f"\n\t{100*power}%", end="")
        
        for proba in probas:
            FPRP = ((1-proba)*alpha)/(((1-proba)*alpha)+(proba*power))
            print(f"\t{100*FPRP:3.1f}%", end="")

### Underestimation of the true variability


In [ ]:
from scipy.stats import chi2

def W(n, alpha=1 - 0.95):
    t2=chi2(df=n-1).ppf(alpha/2)
    t1=chi2(df=n-1).ppf(1-(alpha/2))
    return np.sqrt((n-1)/t1), np.sqrt((n-1)/t2)
    
sample_sizes = [2, 3, 5, 10, 25, 50, 100, 500, 1000]

print("95% CI of standard deviation (SD)")
print("---------------------------------", end='')

for sample_size in sample_sizes:
    print(f"\n{sample_size}", end="")
    print(
        f"\t{W(sample_size)[0]:.2f}×SD - {W(sample_size)[1]:05.2f}×SD",
        end="")

## Conclusion
